# rd_regime_transition и дополнительные гипотезы

**Шаг 2 экспериментов.** Фича rd_regime_transition (момент смены режима) + проверка гипотез для повышения PnL.

Гипотезы:
1. rd_regime_transition — бар, где regime сменился; может быть информативным для входа.
2. Торговля только при transition=1 (переход) vs только при transition=0.
3. Комбинация rd_regime + rd_regime_transition.

## 1. Импорты и загрузка

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import joblib
import warnings
warnings.filterwarnings('ignore')

_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) in ('01_data_prep','02_targets','03_features','04_models','05_experiments') else os.getcwd()
if _root not in sys.path:
    sys.path.insert(0, _root)

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

labeled_path = os.path.join(_root, 'outputs', 'data_labeled_tp_sl_1_05.parquet')
feat_path = os.path.join(_root, 'outputs', 'features_selected_tp_sl_1_05.txt')
if not os.path.exists(labeled_path):
    raise FileNotFoundError(f'Запустите 04_Data_Labeling_And_Feature_Loading.ipynb. Файл: {labeled_path}')
if not os.path.exists(feat_path):
    raise FileNotFoundError(f'Запустите 06_Feature_Selection.ipynb. Файл: {feat_path}')

df = pd.read_parquet(labeled_path)
with open(feat_path, encoding='utf-8') as f:
    FEATURES = [l.strip() for l in f if l.strip()]
FEATURES = [c for c in FEATURES if c in df.columns]

if 'rd_regime_transition' not in df.columns:
    df['rd_regime_prev'] = df.groupby('session_key')['rd_regime'].shift(1)
    df['rd_regime_transition'] = ((df['rd_regime'] != df['rd_regime_prev']) & df['rd_regime_prev'].notna()).astype(int)
    df = df.drop(columns=['rd_regime_prev'], errors='ignore')
if 'rd_regime_transition' not in FEATURES:
    FEATURES.append('rd_regime_transition')

TARGET_COL = 'target'
df['ret_next'] = df.groupby('session_key')['close_price'].pct_change().shift(-1)
valid = df.dropna(subset=FEATURES + [TARGET_COL, 'ret_next']).copy()
valid = valid[valid[TARGET_COL].isin([-1.0, 1.0])]
valid['y'] = (valid[TARGET_COL] == 1).astype(int)
print(f'Строк: {len(valid):,}, rd_regime_transition доля: {valid["rd_regime_transition"].mean():.4f}')

## 2. Сравнение: все фичи vs без rd_regime_transition

In [ ]:
def train_backtest(features, df_valid, commission=0.001, seed=42):
    sessions = df_valid['session_key'].unique().tolist()
    np.random.seed(seed)
    np.random.shuffle(sessions)
    n = len(sessions)
    train_s = set(sessions[:int(0.7*n)])
    test_s = set(sessions[int(0.85*n):])
    train_df = df_valid[df_valid['session_key'].isin(train_s)]
    test_df = df_valid[df_valid['session_key'].isin(test_s)]
    scaler_loc = StandardScaler()
    X_train = scaler_loc.fit_transform(train_df[features].fillna(0))
    X_test = scaler_loc.transform(test_df[features].fillna(0))
    model = lgb.LGBMClassifier(n_estimators=100, max_depth=5, random_state=seed, verbose=-1)
    model.fit(X_train, train_df['y'].values)
    proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(test_df['y'].values, proba)
    bt = test_df.dropna(subset=['ret_next'])
    if len(bt) == 0: return auc, 0.0, 0
    X_bt = scaler_loc.transform(bt[features].fillna(0))
    p = model.predict_proba(X_bt)[:, 1]
    ret = bt['ret_next'].values
    trade = np.where(p >= 0.6, 1, np.where(p <= 0.4, -1, 0))
    pnl = (trade * ret).sum() * 100 - (trade != 0).sum() * commission * 100
    return auc, pnl, (trade != 0).sum()

feat_with = [c for c in FEATURES if c in valid.columns]
feat_no_trans = [c for c in feat_with if c != 'rd_regime_transition']
auc1, pnl1, t1 = train_backtest(feat_with, valid)
auc2, pnl2, t2 = train_backtest(feat_no_trans, valid)
print('С rd_regime_transition: AUC=', f'{auc1:.4f}', 'net PnL=', f'{pnl1:.2f}%', 'trades=', t1)
print('Без rd_regime_transition: AUC=', f'{auc2:.4f}', 'net PnL=', f'{pnl2:.2f}%', 'trades=', t2)

## 3. Гипотеза: торговать только при transition=1

In [ ]:
# Бектест с фильтром: только бары, где rd_regime_transition=1
sessions = valid['session_key'].unique().tolist()
np.random.seed(42)
np.random.shuffle(sessions)
train_s = set(sessions[:int(0.7*len(sessions))])
test_s = set(sessions[int(0.85*len(sessions)):])
train_df = valid[valid['session_key'].isin(train_s)]
test_all = valid[valid['session_key'].isin(test_s)]
test_trans = test_all[test_all['rd_regime_transition'] == 1]
test_no_trans = test_all[test_all['rd_regime_transition'] == 0]

scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[feat_with].fillna(0))
model = lgb.LGBMClassifier(n_estimators=100, max_depth=5, random_state=42, verbose=-1)
model.fit(X_train, train_df['y'].values)

def pnl_subset(subset):
    if len(subset) == 0: return 0.0, 0
    X = scaler.transform(subset[feat_with].fillna(0))
    p = model.predict_proba(X)[:, 1]
    ret = subset['ret_next'].values
    trade = np.where(p >= 0.6, 1, np.where(p <= 0.4, -1, 0))
    return (trade * ret).sum() * 100 - (trade != 0).sum() * 0.001 * 100, (trade != 0).sum()

pnl_all, n_all = pnl_subset(test_all.dropna(subset=['ret_next']))
pnl_tr, n_tr = pnl_subset(test_trans.dropna(subset=['ret_next']))
pnl_ntr, n_ntr = pnl_subset(test_no_trans.dropna(subset=['ret_next']))
print('Backtest (proba>0.6):')
print('  Все бары: net PnL=', f'{pnl_all:.2f}%', 'trades=', n_all)
print('  Только transition=1: net PnL=', f'{pnl_tr:.2f}%', 'trades=', n_tr)
print('  Только transition=0: net PnL=', f'{pnl_ntr:.2f}%', 'trades=', n_ntr)

## 4. Итог шага 11

- rd_regime_transition: влияние на AUC и PnL.
- Гипотеза: фильтр по transition (торговать только при смене режима или только вне смены).

# rd_regime_transition и проверка гипотез

**Шаг 2 плана 06_EXPERIMENTS_PLAN.md.** Проверка влияния rd_regime_transition как фичи, гипотезы: торговать только на переходах, контекст окна, взаимодействия.

## 1. Импорты и загрузка

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) in ('01_data_prep','02_targets','03_features','04_models','05_experiments') else os.getcwd()
if _root not in sys.path:
    sys.path.insert(0, _root)

from src.features.feature_pipeline import get_feature_columns

labeled_path = os.path.join(_root, 'outputs', 'data_labeled_tp_sl_1_05.parquet')
if not os.path.exists(labeled_path):
    raise FileNotFoundError(f'Запустите 04_Data_Labeling_And_Feature_Loading.ipynb. Файл: {labeled_path}')

df = pd.read_parquet(labeled_path)
print(f'Загружено: {len(df):,} строк')
print('rd_regime_transition:', df['rd_regime_transition'].value_counts().to_dict() if 'rd_regime_transition' in df.columns else 'ОТСУТСТВУЕТ')

## 2. Фичи: с rd_regime_transition и без

In [ ]:
feat_path = os.path.join(_root, 'outputs', 'features_selected_tp_sl_1_05.txt')
with open(feat_path, encoding='utf-8') as f:
    FEATURES_BASE = [l.strip() for l in f if l.strip()]
FEATURES_BASE = [c for c in FEATURES_BASE if c in df.columns]
if 'rd_regime_transition' not in FEATURES_BASE and 'rd_regime_transition' in df.columns:
    FEATURES_BASE.append('rd_regime_transition')

FEATURES_WITH_TRANSITION = FEATURES_BASE
FEATURES_WITHOUT_TRANSITION = [c for c in FEATURES_WITH_TRANSITION if c != 'rd_regime_transition']

print(f'С rd_regime_transition: {len(FEATURES_WITH_TRANSITION)} фичей')
print(f'Без rd_regime_transition: {len(FEATURES_WITHOUT_TRANSITION)} фичей')

## 3. Сравнение AUC и PnL: с/без rd_regime_transition

In [ ]:
try:
    import lightgbm as lgb
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import roc_auc_score
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

TARGET_COL = 'target'
COMMISSION = 0.001

def train_eval_backtest(df_in, features, target_col, threshold=0.6, seed=42):
    valid = df_in.dropna(subset=features + [target_col]).copy()
    valid = valid[valid[target_col].isin([-1.0, 1.0])]
    valid['y'] = (valid[target_col] == 1).astype(int)
    
    sessions = valid['session_key'].unique().tolist()
    np.random.seed(seed)
    np.random.shuffle(sessions)
    n = len(sessions)
    n_train, n_val = int(0.7 * n), int(0.15 * n)
    train_sessions = sessions[:n_train]
    test_sessions = sessions[n_train + n_val:]
    
    train_df = valid[valid['session_key'].isin(train_sessions)]
    test_df = valid[valid['session_key'].isin(test_sessions)]
    
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_df[features].fillna(0))
    X_test = scaler.transform(test_df[features].fillna(0))
    
    model = lgb.LGBMClassifier(n_estimators=100, max_depth=5, random_state=seed, verbose=-1)
    model.fit(X_train, train_df['y'].values)
    
    proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(test_df['y'].values, proba)
    
    test_df = test_df.copy()
    test_df['ret_next'] = test_df.groupby('session_key')['close_price'].pct_change().shift(-1)
    bt = test_df.dropna(subset=['ret_next'])
    if len(bt) == 0:
        return auc, 0.0, 0
    X_bt = scaler.transform(bt[features].fillna(0))
    proba_bt = model.predict_proba(X_bt)[:, 1]
    ret = bt['ret_next'].values
    trade = np.where(proba_bt >= threshold, 1, np.where(proba_bt <= 1 - threshold, -1, 0))
    pnl = (trade * ret).sum() * 100 - (trade != 0).sum() * COMMISSION * 100
    return auc, pnl, (trade != 0).sum()

if HAS_LGB:
    auc1, pnl1, n1 = train_eval_backtest(df, FEATURES_WITH_TRANSITION, TARGET_COL)
    auc2, pnl2, n2 = train_eval_backtest(df, FEATURES_WITHOUT_TRANSITION, TARGET_COL)
    print('Сравнение (proba>0.6):')
    print(f'  С rd_regime_transition:  AUC={auc1:.4f}, net PnL={pnl1:.2f}%, trades={n1}')
    print(f'  Без rd_regime_transition: AUC={auc2:.4f}, net PnL={pnl2:.2f}%, trades={n2}')

## 4. Гипотеза: торговать только на переходах rd_regime

In [ ]:
# Стратегия: предсказания модели используем только когда rd_regime_transition == 1
if HAS_LGB:
    valid = df.dropna(subset=FEATURES_WITH_TRANSITION + [TARGET_COL]).copy()
    valid = valid[valid[TARGET_COL].isin([-1.0, 1.0])]
    valid['y'] = (valid[TARGET_COL] == 1).astype(int)
    
    sessions = valid['session_key'].unique().tolist()
    np.random.seed(42)
    np.random.shuffle(sessions)
    n = len(sessions)
    test_sessions = sessions[int(0.85 * n):]
    test_df = valid[valid['session_key'].isin(test_sessions)].copy()
    train_df = valid[valid['session_key'].isin(sessions[:int(0.7 * n)])]
    
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_df[FEATURES_WITH_TRANSITION].fillna(0))
    model = lgb.LGBMClassifier(n_estimators=100, max_depth=5, random_state=42, verbose=-1)
    model.fit(X_train, train_df['y'].values)
    
    test_df['ret_next'] = test_df.groupby('session_key')['close_price'].pct_change().shift(-1)
    bt = test_df.dropna(subset=['ret_next'])
    X_bt = scaler.transform(bt[FEATURES_WITH_TRANSITION].fillna(0))
    proba = model.predict_proba(X_bt)[:, 1]
    ret = bt['ret_next'].values
    
    # Полный бектест
    trade_full = np.where(proba >= 0.6, 1, np.where(proba <= 0.4, -1, 0))
    pnl_full = (trade_full * ret).sum() * 100 - (trade_full != 0).sum() * COMMISSION * 100
    
    # Только на переходах
    trans = bt['rd_regime_transition'].values
    trade_trans = np.where(trans == 1, trade_full, 0)
    pnl_trans = (trade_trans * ret).sum() * 100 - (trade_trans != 0).sum() * COMMISSION * 100
    
    print('Гипотеза: торговать только при rd_regime_transition==1:')
    print(f'  Полный бектест:     net PnL={pnl_full:.2f}%, trades={(trade_full!=0).sum()}')
    print(f'  Только на переходах: net PnL={pnl_trans:.2f}%, trades={(trade_trans!=0).sum()}')

## 5. Гипотеза: rd_regime × volatility (взаимодействие)

In [ ]:
# Добавляем фичу rd_regime * volatility_14
df_hyp = df.copy()
df_hyp['rd_regime_x_vol'] = df_hyp['rd_regime'] * df_hyp['volatility_14']

FEATURES_INTERACT = FEATURES_WITH_TRANSITION + ['rd_regime_x_vol']

if HAS_LGB:
    auc3, pnl3, n3 = train_eval_backtest(df_hyp, FEATURES_INTERACT, TARGET_COL)
    print('С взаимодействием rd_regime × volatility:')
    print(f'  AUC={auc3:.4f}, net PnL={pnl3:.2f}%, trades={n3}')
    print('\nСравнение с baseline (с rd_regime_transition):')
    print(f'  Delta AUC: {auc3 - auc1:+.4f}, Delta PnL: {pnl3 - pnl1:+.2f}%')

## 6. Гипотеза: окно N баров после перехода

In [ ]:
# bars_since_transition: сколько баров прошло с последнего rd_regime_transition
def add_bars_since_transition(df_in, by='session_key'):
    d = df_in.copy()
    def _bars_since(g):
        t = g['rd_regime_transition'].values
        out = np.full(len(g), np.nan, dtype=np.float64)
        last = -999
        for i in range(len(g)):
            if t[i] == 1:
                last = i
            out[i] = i - last if last >= 0 else 999
        return out
    d['bars_since_transition'] = d.groupby(by, group_keys=False).apply(
        lambda g: pd.Series(_bars_since(g), index=g.index)
    )
    return d

df_hyp2 = add_bars_since_transition(df)
FEATURES_BARS = FEATURES_WITH_TRANSITION + ['bars_since_transition']

if HAS_LGB:
    auc4, pnl4, n4 = train_eval_backtest(df_hyp2, FEATURES_BARS, TARGET_COL)
    print('С фичей bars_since_transition:')
    print(f'  AUC={auc4:.4f}, net PnL={pnl4:.2f}%, trades={n4}')
    print('\nСравнение с baseline:')
    print(f'  Delta AUC: {auc4 - auc1:+.4f}, Delta PnL: {pnl4 - pnl1:+.2f}%')

## 7. Итоговая сводка гипотез

In [ ]:
summary = [
    {'hypothesis': 'baseline_with_transition', 'auc': auc1, 'net_pnl': pnl1, 'trades': n1},
    {'hypothesis': 'without_transition', 'auc': auc2, 'net_pnl': pnl2, 'trades': n2},
    {'hypothesis': 'rd_regime_x_volatility', 'auc': auc3, 'net_pnl': pnl3, 'trades': n3},
    {'hypothesis': 'bars_since_transition', 'auc': auc4, 'net_pnl': pnl4, 'trades': n4},
]
summary_df = pd.DataFrame(summary)
out_path = os.path.join(_root, 'outputs', '11_rd_regime_hypotheses_results.csv')
summary_df.to_csv(out_path, index=False)
print(f'Сохранено: {out_path}')
print(summary_df.to_string(index=False))

## Итог шага 2

- Сравнены фичи: с/без rd_regime_transition
- Проверены гипотезы: торговать только на переходах, rd_regime×volatility, bars_since_transition
- Результаты в `outputs/11_rd_regime_hypotheses_results.csv`